# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import clean_html, fetch_website_content

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [3]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [4]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

**GPT:** Welcome! Here’s a quick one:

Why don’t scientists trust atoms? Because they make up everything!

Want a few more options? 
1) Why did the scarecrow win an award? He was outstanding in his field.
2) I would tell you a construction joke, but I’m still working on it.
3) What do you call fake spaghetti? An impasta.

Tell me what kind of humor you like and I’ll tailor some more.

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [5]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

Title: Azam Mustufa — Software Engineer

Azam Mustufa — Software Engineer
AZAM.DEV
Work
Experience
Connect
mail
System Operational · IBM, Bangalore
Software Engineer
at IBM
Software Engineer at IBM working on SAP SuccessFactors cloud integrations. I build backend systems with Java, Spring Boot, and Node.js, and focus on quality engineering, test automation, and distributed systems.
281+
LeetCode · Top 13%
5
Projects deployed
3×
Hackathon finalist
View Work
arrow_downward
Connect
Projects
Backend Systems
terminal
Live
Database Backup CLI Infrastructure
Production-grade automation CLI for async database backups — reduces manual deployment and migration time by 80%, handles 50GB migrations under 150MB memory with 99.9% success rate.
TypeScript
MongoDB
MySQL
Node.js
Linux
Bash
GitHub
arrow_outward
hub
Live
Project Management Engine
Scalable REST backend with 25+ endpoints and granular RBAC. API responses optimised to under 120ms via Redis caching. BullMQ async pipelines cut cache latency b

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [6]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [7]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [8]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [9]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [10]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

**GPT:** - Profile
  - Azam Mustufa, Software Engineer based in Bangalore, open to remote work.
  - Specializes in backend systems, cloud integrations, quality engineering, test automation, and distributed systems.

- Current Role
  - Software Engineer at IBM (Dec 2025 — Present) working on SAP SuccessFactors cloud integrations.
  - Tech: Java, Spring Boot, Spring Data JPA, Hibernate, Spring Security + JWT.
  - Focus: asynchronous processing, transactional data consistency, unit/integration tests.
  - Achievements: resolved 30+ defects during functional testing; improved release stability by ~15%.

- Education & Background
  - B.E. in Computer Science & Engineering, Visvesvaraya Technological University, Belgavi, India.
  - CGPA: 7.28/10; core coursework in OS, networks, OOP, DBMS, DSA, software engineering.
  - 3× Hackathon finalist (regional and university level).

- Projects (selected)
  - Database Backup CLI Infrastructure
    - Production-grade automation CLI for async database backups; 80% time reduction in deployment/migration; handles 50GB migrations with 150MB memory; 99.9% success.
    - Tech: TypeScript, MongoDB, MySQL, Node.js, Linux, Bash.
  - Project Management Engine
    - Scalable REST backend with 25+ endpoints; granular RBAC; Redis caching (<120ms responses); BullMQ pipelines.
    - Tech: Express, MongoDB, Redis, Docker, BullMQ, JWT.
  - Distributed Image Processing Backend
    - Scalable image processing/delivery with dynamic transformations; reduces image payloads by 40%; faster queries with Mongoose; streaming uploads under load.
    - Tech: Bun, Node.js, Express, MongoDB, Mongoose, Sharp.
  - HTTP Caching Proxy
    - CLI-driven proxy caching GET responses to disk; non-GETs forwarded; includes X-Cache header.
    - Tech: TypeScript, Node.js, Express.
  - UPI Offline Mesh
    - Spring Boot backend simulating P2P UPI payments over Bluetooth; encryption with RSA/AES-GCM; atomic idempotency for exactly-once settlement.
    - Tech: Java, Spring Boot, RSA/AES-GCM, H2, JUnit.

- Technical Skills
  - Languages: Java, JavaScript, TypeScript, Python, C++, SQL.
  - Frameworks/Libraries: Spring Boot, Spring Security, Node.js, Express, Bun, BullMQ, Mongoose.
  - Databases/DevOps/Testing: MongoDB, MySQL, Redis, RabbitMQ, Docker, Linux, CI/CD, Git, JUnit, Mockito, Selenium, Playwright, Cypress.

- LeetCode & Hackathons
  - 281+ LeetCode problems, Top 13%.
  - 3× Hackathon finalist.

- Contact & Links
  - Status: Accepting new requests; typically responds within 24 hours.
  - Location: Bangalore, India; open to remote.
  - Resume: View/Download PDF.
  - Links: GitHub, LinkedIn, LeetCode.
  - Email: azammustafa66@gmail.com.

## 6. Reuse the helper on another page

Now that `summarize_website_content` is a function, summarizing a different
site is just: scrape → build messages → call the helper.

In [11]:
content = fetch_website_content(url="https://cnn.com/")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]

In [12]:
summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))

**GPT:** ## CNN homepage summary

- Aimed at delivering breaking news, latest updates, and videos across a wide range of topics. Includes live TV and audio options, plus a personalized experience with account settings and followed topics.
- Core topics you’ll find:
  - US and World news, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather
  - Special topic shelves: World Cup 2026, Ukraine–Russia War, Israel–Hamas War
- Multimedia and formats:
  - Live updates, video clips, analyses, guides
  - CNN 10, podcasts, and a roster of shows (e.g., All There Is with Anderson Cooper, Chasing Life with Dr. Sanjay Gupta)
- Features and series:
  - As Equals, Call to Earth, Freedom Project, Impact Your World, Inside Africa, CNN Heroes
- Regional editions and language options:
  - Editions for US and International audiences; language options include Arabic and Español
- Personalization and tools:
  - Newsletters, topics you follow, sign-in/My Account settings, ad feedback options
- Additional resources:
  - Markets content (Markets Now, pre-markets, after-hours), CNN app download prompts, and broad navigation across sections like Photos, Investigations, and CNN Profiles

If you’d like, I can pull out the top headlines or organize a quick bookmark-friendly summary of the current major stories.

## 7. Selenium-based web scraping

The `requests`-based scraper only sees the initial HTML. For sites that
render their content with JavaScript (single-page apps, infinite scroll,
etc.), we need a real browser. **Selenium** drives a real Chrome instance so
we capture the fully-rendered page, then we reuse `clean_html` to keep the
token count down.

The `WebsiteSummarizer` class below bundles the whole flow — fetch, summarize,
display — behind a small object.

> **Requires:** `selenium` (already in `requirements.txt`). Selenium 4.6+ auto-downloads
> a matching chromedriver, so no manual driver setup is needed — just a local Chrome install.
> A browser window will open while it loads; add `options.add_argument("--headless=new")`
> to run without one.

In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str, base_url: str = "https://api.openai.com/v1", api_key: str | None = None, model: str = "gpt-5-nano") -> None:
        self.url = url
        self.html: str | None = None 
        self.summary: str | None = None 
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        options = Options()
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url=self.url)
            driver.implicitly_wait(5)
            # Strip the rendered HTML down to title + visible text so we send far fewer tokens to the model.
            self.html = clean_html(html=driver.page_source)
        except Exception as e:
            print(f"Error fetching content with Selenium: {e}")
        finally:
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        if not self.html:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        user_prompt = f"Here is the content of the website: {self.html}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,  # type: ignore
        )
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        display(Markdown(data=self.summary))

In [14]:
ws = WebsiteSummarizer(url="https://www.netflix.com/")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

- Netflix India is a streaming service offering ad-free access to a large library of TV shows, movies, documentaries, anime, and Netflix originals.
- Pricing: Plans range from ₹149 to ₹649 per month, with the ability to cancel anytime.
- How to watch: Available on a wide range of devices (smart TVs, consoles, Chromecast, Apple TV, mobiles, web). You can also download titles to watch offline.
- Getting started: Users can sign in or create/restart membership by entering their email.
- Profiles and kids: Supports multiple profiles, including kid-friendly profiles with PIN-protected parental controls.
- Trending Now: The homepage highlights popular titles (e.g., Dhurandhar, Kara, Maa Behen, etc.).
- Key features: Unlimited streaming, offline downloads, ability to save favorites, and watch anywhere across devices.
- What you can watch: A broad library including feature films, documentaries, shows, anime, and Netflix originals.
- Cancel anytime: Flexible cancellation with no penalties.
- Language and accessibility: Language options include English and Hindi; Kids experience is included.
- Help and policy links: FAQs, Help Centre, Terms of Use, Privacy, cookie preferences, and other corporate/contact information are readily available.
- Privacy and cookies: The site uses cookies (Essential, Performance/Functionality, Third-Party, and Advertising cookies) with a Privacy Preference Center and reCAPTCHA protection.

## 8. Point the summarizer at any OpenAI-compatible provider

`WebsiteSummarizer` now takes `base_url`, `api_key`, and `model`, so the exact
same scrape → summarize → display flow can run against a **local model via
Ollama** (or any OpenAI-compatible endpoint) instead of OpenAI — no code
changes, just different constructor arguments.

In [15]:
ws = WebsiteSummarizer(url="https://dhan.co", base_url="http://localhost:11434/v1", api_key="ollama", model="llama3.2:3b")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

It seems like you provided a comprehensive overview of Dhan, an online trading platform in India, along with various legal disclaimers and terms of use. I'll extract some key points from this information:

**Dhan Overview**

* Dhan is an online trading platform that provides a user-first approach for traders and investors.
* It offers various products, including equity trading, futures trading, options trading, commodity trading, ETF investment, and IPO subscription.

**Key Features**

* 6 products within the Dhan ecosystem: Dhan App, Dhan Web, Options Trader App, Options Trader Web, TradingView integration, and Connect to TradingView.
* User can trade on big screens across segments with the web trading platform.
* Dedicated option trading apps for Android and iOS users.

**Important Information**

* Investor awareness and safeguarding client assets: BSE, NSE, and MCX have advisory guidelines for investors.
* Risk management policy and risk disclosure available on the website.
* Terms of usage, disclaimers, privacy policy, grievances, grievance portal, referral terms & conditions, and Saarthi 2.0 mobile app for investors.

**Compliance**

* SEBI registration number: INZ000006031
* Depository participant ID: IN-DP-289-2016
* SEBI research analyst registration number: INH000023357

Please let me know if you'd like me to clarify or expand on any specific points!